In [3]:
import numpy as np
import time
import matplotlib.pyplot as plt
import scipy.stats as stats
import pylab 
import random
import copy

In [4]:
# this class generates systhetic data and fit the sparse ranking model using gradient descent

class Ranking:
    def __init__(self, n, d, k):
        self.n = n
        self.d = d
        self.k = k
    
    def generate_basic(self):
        self.generate_tX()
        self.generate_tb()
        self.score = self.tX @ self.tb
        order = np.argsort(self.score)
        self.ranking = np.zeros(self.n)
        for i in range(self.n):
            self.ranking[order[self.n - 1 - i]] = i + 1
    
    def generate_X(self):
        X = np.random.rand(self.n, self.d) - 0.5
        if True:
            for i in range(self.d):
                X[:, i] = (X[:, i] - np.mean(X[:, i])) / np.std(X[:, i], ddof = 1)
        l = np.max(np.linalg.norm(X, axis=1))
        self.X = X * np.sqrt((1 + self.d) / self.n) / l

    def generate_tX(self):
        self.generate_X()
        tX = np.zeros([self.n, self.n + self.d])
        for i in range(self.n):
            tX[i, i] = 1
            tX[i, self.n : self.n + self.d] = self.X[i, :]
        self.tX = tX
        
    def fix_support(self, v):
        support = (v[:self.n] != 0)
        self.supportsize = int(sum(support))
        self.estimatedsupport = support
        count = 0
        tX_compressed = np.zeros([self.n, self.supportsize + self.d])
        tX_supported = np.zeros([self.n, self.n + self.d])
        for i in range(self.n):
            tX_compressed[i, self.supportsize : self.supportsize + self.d] = self.X[i, :]
            tX_supported[i, self.n : self.n + self.d] = self.X[i, :]
            if support[i]:
                tX_compressed[i, count] = 1
                count += 1
                tX_supported[i, i] = 1
                
        self.tX_compressed = tX_compressed
        self.tX_supported = tX_supported
    
    def soft_thres(self, v, thres):
        return np.sign(v) * np.maximum(np.abs(v) - thres, 0)
        
    def generate_tb(self):
        #alpha = np.random.rand(self.n) * (np.log(5)-1) +0.5
        alpha = np.zeros(self.n)
        support = np.arange(self.k)
        for j in support:
            alpha[j] = (np.random.rand() * (np.log(5)-1) + 1) * np.sign(np.random.randn()) * 0.3
        beta = np.random.randn(self.d)
        beta = beta / np.linalg.norm(beta)
        beta = beta * np.sqrt(self.n / (1 + self.d)) * 0.5
        tb = np.concatenate([alpha, beta], 0)
        self.tb = tb
        self.alpha = self.tb[0 : self.n]
        self.beta = self.tb[self.n : self.n + self.d]
        
        
    def generate_data(self, p, L):
        score = self.tX @ self.tb
        self.graph = np.zeros([self.n, self.n])
        self.y = np.zeros([self.n, self.n])
        self.prob = np.zeros([self.n, self.n])
        self.L = L
        for i in range(self.n):
            for j in range(i):
                if np.random.rand() < p:
                    self.graph[i,j] = 1
                    self.graph[j,i] = 1
                    prob = np.exp(score[j]) / (np.exp(score[i]) + np.exp(score[j]))
                    self.y[i,j] = np.sum(np.random.rand(L) <= prob) / L
                    self.y[j,i] = 1 - self.y[i,j]
                    self.prob[i,j] = prob
                    self.prob[j,i] = 1 - self.prob[i,j]
                    
    def gradient(self, v):
        score = self.tX @ v
        gradient = np.zeros(self.n + self.d)
        for i in range(self.n):
            for j in range(i):
                if self.graph[i,j] == 1:
                    prob = np.exp(score[i]) / (np.exp(score[i]) + np.exp(score[j]))
                    gradient += (prob - self.y[j,i]) * (self.tX[i,:] - self.tX[j,:])
        return gradient
    
    def gradient_supported(self, v):
        score = self.tX @ v
        gradient = np.zeros(self.n + self.d)
        for i in range(self.n):
            for j in range(i):
                if self.graph[i,j] == 1:
                    prob = np.exp(score[i]) / (np.exp(score[i]) + np.exp(score[j]))
                    gradient += (prob - self.y[j,i]) * (self.tX_supported[i,:] - self.tX_supported[j,:])
        return gradient
    
    def Hessian(self, v):
        score = self.tX @ v
        H = np.zeros([self.n + self.d, self.n + self.d])
        for i in range(self.n):
            for j in range(i):
                if self.graph[i,j] == 1:
                    coeff = np.exp(score[i] + score[j]) / ((np.exp(score[i]) + np.exp(score[j])) ** 2)
                    contrast = (self.tX[i,:]-self.tX[j,:])[np.newaxis]
                    H += coeff * contrast.T * contrast
        return H
    
    def compressedH(self, v):
        H = self.Hessian(v)
        H_compressed = np.zeros([self.supportsize + self.d, self.supportsize + self.d])
        counti = 0
        for i in range(self.n):
            if self.estimatedsupport[i]:
                countj = 0
                for j in range(self.n):
                    if self.estimatedsupport[j]:
                        H_compressed[counti, countj] = H[i, j]
                        countj += 1
                H_compressed[counti, self.supportsize:] = H[i, self.n:]
                
                counti += 1
                
        for i in range(self.d):
            countj = 0
            for j in range(self.n):
                #print(self.supportsize + i, countj, j)
                if self.estimatedsupport[j]:
                    H_compressed[self.supportsize + i, countj] = H[self.n + i, j]
                    countj += 1
            H_compressed[self.supportsize + i, self.supportsize:] = H[self.n + i, self.n:]

        return H_compressed
    
    def diamondinv(self, v):
        H = self.Hessian(v)
        diamond = np.zeros([self.n + self.d, self.n + self.d])
        for i in range(self.n):
            diamond[i, i] = 1 / H[i, i]
        diamond[self.n:, self.n:] = np.linalg.inv(H[self.n:, self.n:])
        return diamond
    
    def generate_MLE(self, tau = 5e-4, lamb = 5):
        eta = 0.001
        v_prev1 = np.zeros(self.n + self.d)
        v_prev2 = np.zeros(self.n + self.d)
        for i in range(10000):
            tmp = v_prev1 + (i / (i + 3)) * (v_prev1 - v_prev2) 
            gradient = self.gradient(tmp)
            v = (1 - eta * tau) * tmp - eta * gradient
            v[:self.n] = np.sign(v[:self.n]) * np.maximum(0, np.abs(v[:self.n]) - eta * lamb)
            if np.linalg.norm(v - v_prev1) < 1e-5:
                return v
            else:
                #print(i, np.linalg.norm(v - v_prev1))
                v_prev2 = v_prev1
                v_prev1 = v
        return 
        
    def generate_MLE_support(self):
        eta = 0.001
        v_prev = np.zeros(self.n + self.d)
        for i in range(10000):
            gradient = self.gradient_supported(v_prev)
            v = v_prev - eta * gradient
            if np.linalg.norm(v - v_prev) < 1e-5:
                return v
            else:
                v_prev = v
        return 

    def debias_test(self, v, m = 0):
        g = self.gradient(v)
        H = self.Hessian(v)
        result = v[m] - (g[m] / H[m, m])
        return (result - self.tb[m]) * np.sqrt(H[m, m] * self.L)
    
    def beta_test(self, v, m = 0):
        H = self.Hessian(v)
        A_inverse = np.linalg.inv(H[self.n:, self.n:])
        return (v[self.n + m] - self.tb[self.n + m]) * np.sqrt(self.L / A_inverse[m, m])
    
    def test_both(self, v, m1 = 0, m2 = 0):
        g = self.gradient(v)
        H = self.Hessian(v)
        result = v[m1] - (g[m1] / H[m1, m1])
        A_inverse = np.linalg.inv(H[self.n:, self.n:])
        return (result - self.tb[m1]) * np.sqrt(H[m1, m1] * self.L), (v[self.n + m2] - self.tb[self.n + m2]) * np.sqrt(self.L / A_inverse[m2, m2])
    
    def error(self, v):
        return np.max(np.abs(v[:self.n] - self.alpha)), np.linalg.norm(v[self.n:] - self.beta) / np.linalg.norm(self.beta)
    
    def GMB2(self, v, num, m = 0):
        #H = self.Hessian(v)
        H_compressed = self.compressedH(v)
        H_inv = np.linalg.inv(H_compressed)
            
        sigma_recorded = np.zeros(self.n)
        up_all_recorded = np.zeros([self.n, self.n])
        scores_recorded = self.tX @ v
        for k in range(self.n):
            if k != m:
                s = 0
                diff = self.tX_compressed[m, :] - self.tX_compressed[k, :]
                tmp = H_inv @ diff
                sigma_recorded[k] = np.sqrt(diff.T @ H_inv @ diff / self.L)
                up_all_recorded[k, :] = tmp @ self.tX_compressed.T
        results = []
        for it in range(num):
            print(it)
            result = -1
            hada = np.zeros([self.n, self.n])
            for i in range(self.n):
                for j in range(i):
                    tmp = scores_recorded[i] - scores_recorded[j]
                    tmp = np.exp(tmp) / (1 + np.exp(tmp))
                    hada[i, j] = (tmp - 1) * np.random.randn() * np.sqrt(self.y[j, i] * self.L) 
                    hada[i, j] += tmp * np.random.randn() * np.sqrt(self.y[i, j] * self.L)
                    hada[i, j] /= self.L
            for k in range(self.n):
                if k != m:
                    s = 0
                    for i in range(self.n):
                        for j in range(i):
                            up = up_all_recorded[k, i] - up_all_recorded[k, j]
                            s += up * hada[i, j] / sigma_recorded[k]
                    if np.abs(s) > result:
                        result = np.abs(s)
            results.append(result)
            
        return results
 
    def CI2(self, v, m = 0):
        tmp = self.GMB2(v, 200, m)
        tmp.sort()
        c = (tmp[-10] + tmp[-11]) / 2
        
        H_compressed = self.compressedH(v)
        H_inv = np.linalg.inv(H_compressed)
        result = np.zeros([self.n - 1, 2])
        count = 0
        T = 0
        for k in range(self.n):
            if k != m:
                diff = self.tX_compressed[m, :] - self.tX_compressed[k, :]
                sigma = np.sqrt(diff.T @ H_inv @ diff / self.L)
                
                thetadiff = v @ (self.tX[k, :] - self.tX[m, :])
                result[count, 0] = thetadiff - c * sigma
                result[count, 1] = thetadiff + c * sigma
                ABS = np.abs(thetadiff - (self.score[k] - self.score[m])) / sigma
                if ABS > T:
                    T = ABS
                    
                count += 1
                
        l = 1
        r = self.n
        for i in range(self.n - 1):
            if result[i, 0] > 0 and result[i, 1] > 0:
                l += 1
            if result[i, 0] < 0 and result[i, 1] < 0:
                r -= 1
                    
        return l, r, self.ranking[m], T, c
    
    def GMB1(self, v, num, m = 0):
        diamond = self.diamondinv(v)
        H = self.Hessian(v)
            
        sigma_recorded = np.zeros(self.n)
        up_all_recorded = np.zeros([self.n, self.n])
        scores_recorded = self.tX @ v
        for k in range(self.n):
            if k != m:
                s = 0
                diff = self.tX[m, :] - self.tX[k, :]
                tmp = diamond @ diff
                sigma_recorded[k] = np.sqrt(tmp.T @ H @ tmp / self.L)
                up_all_recorded[k, :] = tmp @ self.tX.T
        results = []
        for it in range(num):
            print(it)
            result = -1
            hada = np.zeros([self.n, self.n])
            for i in range(self.n):
                for j in range(i):
                    tmp = scores_recorded[i] - scores_recorded[j]
                    tmp = np.exp(tmp) / (1 + np.exp(tmp))
                    hada[i, j] = (tmp - 1) * np.random.randn() * np.sqrt(self.y[j, i] * self.L) 
                    hada[i, j] += tmp * np.random.randn() * np.sqrt(self.y[i, j] * self.L)
                    hada[i, j] /= self.L
            for k in range(self.n):
                if k != m:
                    s = 0
                    for i in range(self.n):
                        for j in range(i):
                            up = up_all_recorded[k, i] - up_all_recorded[k, j]
                            s += up * hada[i, j] / sigma_recorded[k]
                    if np.abs(s) > result:
                        result = np.abs(s)
            results.append(result)
            
        return results
 
    def CI1(self, v, m = 0):
        tmp = self.GMB1(v, 200, m)
        tmp.sort()
        c = (tmp[-10] + tmp[-11]) / 2
        
        diamond = self.diamondinv(v)
        H = self.Hessian(v)
        g = self.gradient(v)
        result = np.zeros([self.n - 1, 2])
        count = 0
        T = 0
        for k in range(self.n):
            if k != m:
                diff = self.tX[m, :] - self.tX[k, :]
                tmp = diamond @ diff
                sigma = np.sqrt(tmp.T @ H @ tmp / self.L)
                
                thetadiff = (v[k] - (g[k] / H[k, k])) - (v[m] - (g[m] / H[m, m]))
                thetadiff += v[self.n:] @ (self.X[k, :] - self.X[m, :])
                result[count, 0] = thetadiff - c * sigma
                result[count, 1] = thetadiff + c * sigma
                ABS = np.abs(thetadiff - (self.score[k] - self.score[m])) / sigma
                if ABS > T:
                    T = ABS
                    
                count += 1
                
        l = 1
        r = self.n
        for i in range(self.n - 1):
            if result[i, 0] > 0 and result[i, 1] > 0:
                l += 1
            if result[i, 0] < 0 and result[i, 1] < 0:
                r -= 1
                    
        return l, r, self.ranking[m], T, c

In [ ]:
# generate simulation results under each setting for 500 times

In [ ]:
debiased = np.zeros(500)
beta = np.zeros(500)
for i in range(500):
    print(i)
    data = Ranking(200, 3, 5)
    data.generate_basic()
    data.generate_data(0.5, 25)
    v = data.generate_MLE(lamb = 1)
    tmp = data.test_both(v)
    debiased[i] = tmp[0]
    beta[i] = tmp[1]
    np.save('./sparserankingsimulation/normalityalphaJan8L25lamb1', debiased)
    np.save('./sparserankingsimulation/normalitybetaJan8L25lamb1', beta)

0


In [ ]:
debiased = np.zeros(500)
beta = np.zeros(500)
for i in range(500):
    print(i)
    data = Ranking(200, 3, 5)
    data.generate_basic()
    data.generate_data(0.5, 25)
    v = data.generate_MLE(lamb = 3)
    tmp = data.test_both(v)
    debiased[i] = tmp[0]
    beta[i] = tmp[1]
    np.save('./sparserankingsimulation/normalityalphaJan8L25lamb3', debiased)
    np.save('./sparserankingsimulation/normalitybetaJan8L25lamb3', beta)

In [ ]:
debiased = np.zeros(500)
beta = np.zeros(500)
for i in range(500):
    print(i)
    data = Ranking(200, 3, 5)
    data.generate_basic()
    data.generate_data(0.1, 10)
    v = data.generate_MLE(lamb = 1.2)
    tmp = data.test_both(v)
    debiased[i] = tmp[0]
    beta[i] = tmp[1]
    np.save('./sparserankingsimulation/normalityalphaJan8L10lamb12', debiased)
    np.save('./sparserankingsimulation/normalitybetaJan8L10lamb12', beta)